# 06 — Voice Cloning: Speaker Embedding + Synthesis (Kaggle / Colab / Local)

Zero-shot voice cloning dùng XTTS-v2:
- Extract speaker embedding từ 5-30s reference audio
- Synthesize utterances mới với giọng clone
- Đánh giá speaker similarity (cosine similarity)

**Checkpoint**: Tự động tìm best model từ fine-tuned hoặc fallback sang base XTTS-v2.

In [ ]:
# ── Cell 0: Environment + Paths (Kaggle / Colab / Local) ─────────────────────
import sys, os
from pathlib import Path

if os.path.exists('/kaggle/working/project/training/notebooks'):
    sys.path.insert(0, '/kaggle/working/project/training/notebooks')

from _env import *

MODEL_DIR  = os.path.join(WEIGHTS_DIR, 'xtts-v2')
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'storage', 'voice_clones')
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'ENV        : {ENV_NAME}')
print(f'Device     : {device}')
print(f'MODEL_DIR  : {MODEL_DIR}')
print(f'AUDIO_DIR  : {AUDIO_DIR}')
print(f'OUTPUT_DIR : {OUTPUT_DIR}')

if IS_KAGGLE:
    print('\n⚠️  Kaggle: Output lưu vào /kaggle/working/ — Save Version để persist!')

In [ ]:
# ── Cell 1: Cài dependencies ──────────────────────────────────────────────────
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'TTS', 'soundfile', 'scikit-learn'], check=False)
print('✓ Dependencies installed')

In [ ]:
# ── Cell 2: Load XTTS-v2 ──────────────────────────────────────────────────────
from TTS.api import TTS

model_pth   = os.path.join(MODEL_DIR, 'model.pth')
config_json = os.path.join(MODEL_DIR, 'config.json')

if os.path.exists(model_pth) and os.path.exists(config_json):
    print('Loading fine-tuned XTTS-v2...')
    tts = TTS(model_path=model_pth, config_path=config_json).to(device)
    print('✅ Fine-tuned XTTS-v2 loaded')
else:
    print('Fine-tuned model không tìm thấy → dùng base XTTS-v2...')
    tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
    print('✅ Base XTTS-v2 loaded')

In [ ]:
# ── Cell 3: Chọn reference audio ──────────────────────────────────────────────
import glob, numpy as np, soundfile as sf, tempfile

reference_path = None

if os.path.isdir(AUDIO_DIR):
    wav_files = glob.glob(os.path.join(AUDIO_DIR, '*.wav'))
    for wf in wav_files:
        try:
            info = sf.info(wf)
            if info.duration >= 5.0:
                reference_path = wf
                print(f'✓ Reference: {os.path.basename(wf)} ({info.duration:.1f}s)')
                break
        except Exception:
            pass
    if reference_path is None and wav_files:
        reference_path = wav_files[0]
        print(f'✓ Reference: {os.path.basename(reference_path)}')

if reference_path is None:
    tmpdir = tempfile.gettempdir()
    reference_path = os.path.join(tmpdir, 'reference_vietspeech.wav')
    sf.write(reference_path, np.random.randn(10 * 16000).astype(np.float32) * 0.01, 16000)
    print(f'⚠️  Dùng dummy reference: {reference_path}')

texts_to_clone = [
    'Xin chào, tôi đang nói chuyện với giọng được sao chép.',
    'Hệ thống nhận dạng tiếng Việt đa phương ngữ.',
    'Tôi sống ở Hà Nội và học tại trường đại học.',
]

In [ ]:
# ── Cell 4: Voice Cloning ──────────────────────────────────────────────────────
from tqdm.auto import tqdm

print(f'Synthesizing {len(texts_to_clone)} utterances...')
for i, text in enumerate(tqdm(texts_to_clone, desc='Cloning')):
    output_path = os.path.join(OUTPUT_DIR, f'clone_{i+1}.wav')
    try:
        tts.tts_to_file(
            text=text, language='vi',
            speaker_wav=reference_path, file_path=output_path,
        )
        size_kb = os.path.getsize(output_path) / 1024
        print(f'  ✓ clone_{i+1}.wav ({size_kb:.0f}KB)')
    except Exception as e:
        print(f'  ❌ clone_{i+1} lỗi: {e}')

print(f'\nOutput dir: {OUTPUT_DIR}')

In [ ]:
# ── Cell 5: Speaker Similarity Evaluation ─────────────────────────────────────
from sklearn.metrics.pairwise import cosine_similarity

def extract_speaker_embedding(audio_path, tts_model):
    gpt_cond_latent, speaker_embedding = \
        tts_model.synthesizer.tts_model.get_conditioning_latents(audio_path=[audio_path])
    return speaker_embedding.cpu().numpy().flatten()

try:
    ref_embed = extract_speaker_embedding(reference_path, tts)
    print('Speaker Similarity (cosine vs. reference):')
    all_pass = True
    for i in range(len(texts_to_clone)):
        cp = os.path.join(OUTPUT_DIR, f'clone_{i+1}.wav')
        if os.path.exists(cp):
            sim    = cosine_similarity([ref_embed], [extract_speaker_embedding(cp, tts)])[0][0]
            status = '✅ PASS' if sim >= 0.90 else '❌ FAIL'
            if sim < 0.90: all_pass = False
            print(f'  clone_{i+1}.wav: {sim:.3f} {status} (target ≥ 0.90)')
    print(f'\nOverall: {"✅ ALL PASS" if all_pass else "⚠️  Some below target"}')
except Exception as e:
    print(f'⚠️  Evaluation lỗi: {e}')

print('\n✓ Voice cloning complete')

if IS_KAGGLE:
    print('\n⚠️  Kaggle: Nhấn Save Version để lưu output!')

## ✅ Voice Cloning Demo Complete

| Platform | Output location |
|----------|-----------------|
| Kaggle | `/kaggle/working/storage/voice_clones/` |
| Colab | `Drive/GenAI_ViMD/storage/voice_clones/` |

**Next**: Run `07_export_onnx.ipynb`